# Solar Power Generation Data Analysis
## Dataset Understanding and Exploration

A practical session on dataset understanding and exploration using real solar power generation data. 

### Learning Objectives:
- Understand the structure and characteristics of the datasets
- Perform exploratory data analysis (EDA) on solar power data
- Identify patterns, trends, and anomalies in renewable energy generation
- Learn data quality assessment techniques
- Visualize energy data effectively


## About Dataset
This data has been gathered at two solar power plants in India over a 34 day period. It has two pairs of files - each pair has one power generation dataset and one sensor readings dataset. The power generation datasets are gathered at the inverter level - each inverter has multiple lines of solar panels attached to it. The sensor data is gathered at a plant level - single array of sensors optimally placed at the plant.

https://www.kaggle.com/datasets/anikannal/solar-power-generation-data/data

## 💡 Power (kW) vs. Energy (kWh)

In the energy sector, it is crucial to distinguish between two closely related concepts that have distinct physical and economic meanings: **Power** and **Energy**.

### 1. Power (kW - Kilowatt)

Power is a measure of the **rate of energy production or consumption at a given instant** (or the average rate over a short interval).

* **Physical Definition:** It is the "speed" at which energy is delivered.
* **Unit of Measure:** $\text{kW}$ (Kilowatt)
* **Business Meaning:** Represents the instantaneous **capacity** or operational output of an asset. It is vital for **network management** and **real-time optimization** (e.g., Trading).

### 2. Energy (kWh - Kilowatt-hour)

Energy is a measure of the **total amount of work performed** (or energy generated/consumed) over a period of time.

* **Physical Definition:** It is Power supplied (or consumed) *multiplied by time*.
* **Unit of Measure:** $\text{kWh}$ (Kilowatt-hour)
* **Business Meaning:** Represents the total **quantity** that is sellable or usable. It is the unit that is **billed** to customers and used to calculate total production **revenue**.

### The Relationship:

Energy is the result of integrating Power over time:

$$\text{Energy } (\text{kWh}) = \text{Power } (\text{kW}) \times \text{Time } (\text{hours})$$

> **Practical Example:** If our inverter registers in average $100 \text{ kW}$ of power in a $15\text{-minute } (0.25 \text{ hour})$ interval, the total energy generated for billing is $100 \text{ kW} \times 0.25 \text{ h} = 25 \text{ kWh}$.

In [ ]:
# Import Required Libraries
import os
import pandas as pd
import numpy as np
from datetime import date, time
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings

# Configure visualization settings
plt.style.use('default')
sns.set_palette("husl")
warnings.filterwarnings('ignore')

# Display options for better data viewing
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 100)

print("Libraries imported successfully")

## 1. Data Import and Initial Inspection

In this section, we'll import our solar power datasets and perform initial data inspection to understand:
- Dataset structure and dimensions
- Column names and data types
- Missing values
- Basic statistical summaries


### 1.1 Loading the Datasets

In [ ]:
def load_data(data_folder="../data", generation_file="Plant_1_Generation_Data.csv", weather_file="Plant_1_Weather_Sensor_Data.csv", ):
        
    generation_path = os.path.join(data_folder, generation_file)
    weather_path = os.path.join(data_folder, weather_file)

    generation_df = pd.read_csv(generation_path)
    weather_df = pd.read_csv(weather_path)

    return generation_df, weather_df

generation_df, weather_df = load_data()

# Check datasets size
print(f"Generation Data Shape: {generation_df.shape}")
print(f"Weather Data Shape: {weather_df.shape}")

### 1.2 Initial Data Exploration

Let's examine the structure and basic characteristics of our datasets.

In [ ]:
# Info


In [ ]:
# Convert DATE_TIME to datetime format


# Create DAY and DAYTIME columns


In [ ]:
# Look at the dataset head


In [ ]:
# Summary statistics


##### We have 22 sources for 34 days and 96 sample per day (quarter of hour)

##### What about the weather dataset?

In [ ]:
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'], format='%Y-%m-%d %H:%M:%S')
weather_df['DAY'] = weather_df['DATE_TIME'].dt.date
weather_df['DAYTIME'] = weather_df['DATE_TIME'].dt.time
weather_df.describe(include='all')

##### In weathern dataset we have 1 source for 34 days and 96 sample per day

## 2. Univariate Data Exploration

In this section, we'll explore individual variables to understand their distributions, identify outliers, and gain insights into the characteristics of each feature.

### 2.1 Timestamp: do we have a complete dataset?

Let's examine the completeness of the dataset


In [ ]:
# Plot number of records per day


##### Some day we don't have all the data, it should ben 22*96 = 2112 sample per day

In [ ]:
# Hom may source with complete data?
total_sample_per_day = 96
source_dataquality = generation_df.groupby(['DAY','SOURCE_KEY'])[['DAYTIME']].count().rename({'DAYTIME': 'sample_completeness'})
source_dataquality['sample_completeness'] = round(source_dataquality/total_sample_per_day * 100, 2)

In [ ]:
# Plot a heatmap with SOURCE_KEY on rows and DAY on cols
heatmap_data = source_dataquality.reset_index().pivot(index='SOURCE_KEY', columns='DAY', values='sample_completeness')

# Create the heatmap
plt.figure(figsize=(16, 10))
sns.heatmap(heatmap_data, 
            annot=False,  # Set to True if you want to show values in cells
            cmap='RdYlGn',  # Red-Yellow-Green colormap (red=bad, green=good)
            center=90,  # Center the colormap at 75% (middle of 50-100 range)
            vmin=60, vmax=100,  # Set range from 50% to 100%
            cbar_kws={'label': 'Data Completeness (%)'})

plt.title('Data Completeness Heatmap: SOURCE_KEY vs DAY', fontsize=16, fontweight='bold')
plt.xlabel('Day', fontsize=12)
plt.ylabel('SOURCE_KEY', fontsize=12)
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Print summary statistics
print("\n=== DATA COMPLETENESS SUMMARY ===")
print(f"Overall average completeness: {heatmap_data.mean().mean():.2f}%")
print(f"Minimum completeness: {heatmap_data.min().min():.2f}%")
print(f"Maximum completeness: {heatmap_data.max().max():.2f}%")
print(f"Number of 100% complete combinations: {(heatmap_data == 100).sum().sum()}")
print(f"Number of missing combinations: {heatmap_data.isna().sum().sum()}")

##### At this point you should go back to business understanding phase and ask for following questions:

- What happened on 21 and 29 of may? 
- Missing values describe a malfunction of the inverter or a problem on data stream?
- Can we retrive missing data? If not, do we have any other info to reconstruct the data?

